## Re-Ranking Techniques

Re-Ranking is a second-stage filtering process in retrieval systems,especially in RAG pipelines where we:

1. First we use a retriever to get top k documents.
2. Then use a more accurate but slower model like LLM to re-score and reorder those elements by relevance to the query

In [110]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [111]:

loaded_data=TextLoader("topic.txt",encoding="utf-8")
docs=loaded_data.load()

text_splitters=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=text_splitters.split_documents(docs)

In [112]:
print(f"generated :{len(chunks)}")
chunks

generated :89


[Document(metadata={'source': 'topic.txt'}, page_content='================================================================================\nTOPIC CORPUS FOR RE-RANKING PRACTICE\n================================================================================\nTheme: Grid-scale energy storage, batteries, and power-system stability.'),
 Document(metadata={'source': 'topic.txt'}, page_content='How to use this file:\n- Split on "=== CHUNK-" boundaries to build a retrieval index.\n- First-stage retrieval (BM25 / dense) will often return plausible but wrong\n  chunks because many passages share words: battery, grid, MW, lithium,\n  frequency, inverter, capacity, power, voltage, thermal, cycle, efficiency.\n- A re-ranker (cross-encoder, LLM judge, or learning-to-rank features) should\n  prefer chunks whose *claims and constraints* match the query, not just\n  lexical overlap.'),
 Document(metadata={'source': 'topic.txt'}, page_content='Suggested practice queries appear at the end under QUERY 

In [113]:
embedings=OpenAIEmbeddings(model="text-embedding-3-small")
dense_vectorstore=FAISS.from_documents(chunks,embedding=embedings)

In [114]:
sparse_retriever=BM25Retriever.from_documents(chunks)
sparse_retriever.k=9

dense_retriever=dense_vectorstore.as_retriever(k=9)

In [115]:
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weights=[0.7,0.3]
)

In [116]:
hybrid_retriever.invoke("Explain why lithium-ion batteries can respond to grid frequency deviations  faster than many traditional thermal generators.")

[Document(id='3d8b8d9d-ac88-464c-8d19-871e3560afaf', metadata={'source': 'topic.txt'}, page_content='=== CHUNK-075 ===\nTitle: Detailed answer: why batteries help frequency quickly\nWhen load suddenly exceeds generation, synchronous inertia slows the rate of\nfrequency decline, but additional active power must be injected to restore\nbalance. Lithium-ion battery inverters can change real power output within\nhundreds of milliseconds to seconds, making them effective for fast frequency\nresponse products. The limitation is energy: rapid power injection depletes'),
 Document(id='f74e3ebe-4e13-41ce-8907-c55f85ca528b', metadata={'source': 'topic.txt'}, page_content="=== CHUNK-087 ===\nTitle: Interview-style Q/A mimic\nQ: Why can't we use only thermal plants for fast frequency events?\nA: Thermal steam plants have inertia and can ramp, but many cannot change\noutput as quickly as batteries for the first seconds after an imbalance. Markets\ntherefore price fast response separately from slowe

In [117]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000278AA7D2490>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000278AA6397B0>, k=9)], weights=[0.7, 0.3])

In [118]:
# import os
# llm=ChatGroq(model="groq:gemma2-9b-it")
# os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm=ChatOpenAI(model="gpt-3.5-turbo")

In [119]:
prompt=PromptTemplate.from_template("""
You are a helpful assistant. Your taskis to rannk the followinf documents from most to least relevant.
                                    
user Question:{question}
                                    
Documents:
{documents}

                                    Instructions:
                                    -think about the relevance of each document to user's question
                                    - return a list of couments indices in ranked order,starting from the most relevant

                                    output format:
                                    comma separated document indices (e.g. 2,1,3,0)
                                                                       
""")

In [120]:
retrieved_docs=hybrid_retriever.invoke("We want 95% round-trip efficiency for our battery project.")
retrieved_docs

[Document(id='912bee2d-2950-457b-afc5-af6650d93f44', metadata={'source': 'topic.txt'}, page_content='=== CHUNK-081 ===\nTitle: Contradiction pair setup (dated claim style)\nEarly blog posts sometimes claimed round-trip battery efficiency "above 95%"\nas a universal rule. Modern station-level reporting often shows lower realized\nefficiency after HVAC, transformer, and auxiliary loads—especially at partial\nload.'),
 Document(id='9f985932-9f57-4a62-91cc-0c3d5ac3f18d', metadata={'source': 'topic.txt'}, page_content='=== CHUNK-082 ===\nTitle: Contradiction pair setup (nuanced claim style)\nRealized AC-to-AC round-trip efficiency for utility battery stations depends on\noperating point, temperature, and auxiliary loads; headline DC cell efficiency\nis only one component. Treat single-number efficiency claims with skepticism\nunless the boundary conditions are defined.'),
 Document(id='8104c901-c4d3-4cfc-b9df-039dbb27ef7c', metadata={'source': 'topic.txt'}, page_content='=== CHUNK-008 ===\n

In [121]:
doc_lines=[f"{i+1}. {doc.page_content}" for i,doc in enumerate(retrieved_docs)]
formatted_docs="\n".join(doc_lines)

In [122]:
doc_lines

['1. === CHUNK-081 ===\nTitle: Contradiction pair setup (dated claim style)\nEarly blog posts sometimes claimed round-trip battery efficiency "above 95%"\nas a universal rule. Modern station-level reporting often shows lower realized\nefficiency after HVAC, transformer, and auxiliary loads—especially at partial\nload.',
 '2. === CHUNK-082 ===\nTitle: Contradiction pair setup (nuanced claim style)\nRealized AC-to-AC round-trip efficiency for utility battery stations depends on\noperating point, temperature, and auxiliary loads; headline DC cell efficiency\nis only one component. Treat single-number efficiency claims with skepticism\nunless the boundary conditions are defined.',
 '3. === CHUNK-008 ===\nTitle: Round-trip efficiency comparison (hand-wavy)\nPeople sometimes say "batteries are ~90% efficient" while "hydro is ~80%."\nThese numbers are not universally comparable because they ignore auxiliary\nplant loads, transformer losses, and the difference between DC cell efficiency\nand A

In [123]:
formatted_docs

'1. === CHUNK-081 ===\nTitle: Contradiction pair setup (dated claim style)\nEarly blog posts sometimes claimed round-trip battery efficiency "above 95%"\nas a universal rule. Modern station-level reporting often shows lower realized\nefficiency after HVAC, transformer, and auxiliary loads—especially at partial\nload.\n2. === CHUNK-082 ===\nTitle: Contradiction pair setup (nuanced claim style)\nRealized AC-to-AC round-trip efficiency for utility battery stations depends on\noperating point, temperature, and auxiliary loads; headline DC cell efficiency\nis only one component. Treat single-number efficiency claims with skepticism\nunless the boundary conditions are defined.\n3. === CHUNK-008 ===\nTitle: Round-trip efficiency comparison (hand-wavy)\nPeople sometimes say "batteries are ~90% efficient" while "hydro is ~80%."\nThese numbers are not universally comparable because they ignore auxiliary\nplant loads, transformer losses, and the difference between DC cell efficiency\nand AC stati

In [124]:
chain=prompt|llm|StrOutputParser()
response=chain.invoke({"question":"A vendor calls a fuel-cell backup product a 'grid battery.' Why might planners object to that wording?","documents":formatted_docs})

In [125]:
response

'6,7,1,3,2,9,4,5,8'